# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Method:** this cell makes sure DuckDB is version 1.1 or newer, the first version whose Hugging Face login can read the token from the environment.


In [ ]:
%pip -q install "duckdb>=1.1"

## Setup

**Method, step by step:**

1. **Finds the project folder** and **reads the Hugging Face token**, as in the data contract: environment variable or `.env` file, then a Colab Secret named `HF_TOKEN`, then a typed prompt as the last resort. DuckDB reads the token from the environment, so it never appears in a query or an output.
2. **Loads the tools.** `duckdb` queries the warehouse, `pandas` and `numpy` handle the results, `scipy` supplies the negative binomial luck test behind the target and rule B, `scikit-learn` supplies logistic regression, the random forest used in the paper audit, the fold splitters, and the metrics, and `matplotlib` draws the one chart.
3. **Defines the months and the cache.** The same three development pairs as the data contract; June 2026 is never read. Heavy queries go through `cached()`, which saves each result once under `work/outputs/w03_cache/` and reuses it on later runs.
4. **Defines `show()` and `show_table()`**, which display every result as a titled table.


In [ ]:
import getpass
import os
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import nbinom
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, KFold

# Locally, walk up to the project's top folder (the one holding data/raw and .env); Colab keeps its own folder.
if "google.colab" not in sys.modules:
    while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")


def load_env_file(path: str = ".env") -> None:
    """Copy KEY=value lines from a local .env file into the environment, keeping any value already set."""
    if not os.path.exists(path):
        return
    with open(path, encoding="utf-8-sig") as handle:
        for line in handle:
            key, sep, value = line.strip().partition("=")
            if sep and key and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip())


def read_colab_secret(name: str) -> str | None:
    """Return the Colab Secret called `name`, or None when it cannot be read.

    Outside Colab the import fails. On Colab the secret can be missing, not shared with this
    notebook, or the request can time out. Each case returns None so the caller can ask for the token.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        return None


# Token order: environment variable or .env file, then Colab Secret, then a typed prompt as the last resort.
load_env_file()
token_source = "environment variable or .env file"
if not os.environ.get("HF_TOKEN"):
    secret = read_colab_secret("HF_TOKEN")
    token_source = "Colab Secret" if secret else "typed prompt"
    os.environ["HF_TOKEN"] = secret or getpass.getpass("Hugging Face READ token (hf_...): ")

# DuckDB reads the token from the environment, so it never appears in a query or an output.
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, PROVIDER credential_chain)")
# Cap DuckDB's memory so large scans spill to disk, and hide its progress bars, which clutter saved outputs.
con.execute("SET memory_limit = '2GB'")
con.execute("SET preserve_insertion_order = false")
con.execute("SET enable_progress_bar = false")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
QUERY_90D = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
DAILY_FOOTERS = f"read_parquet('{REL}/fact_content_daily_performance/*/*.parquet')"
PAIRS = [("2026-02", "2026-03"), ("2026-03", "2026-04"), ("2026-04", "2026-05")]
DEV_MONTHS = sorted({month for pair in PAIRS for month in pair})
PANEL_MONTHS = list(pd.period_range("2025-01", "2026-05", freq="M").astype(str))
assert "2026-06" not in DEV_MONTHS + PANEL_MONTHS, "The sealed month must never be read."


def month_files(months: list[str]) -> str:
    """Return a DuckDB source that reads exactly the listed monthly partitions of the daily table."""
    paths = ", ".join(f"'{REL}/fact_content_daily_performance/month={month}/*.parquet'" for month in months)
    return f"read_parquet([{paths}])"


CACHE = "work/outputs/w03_cache"
os.makedirs(CACHE, exist_ok=True)


def cached(name: str, sql: str) -> pd.DataFrame:
    """Run a warehouse query once, save its result as Parquet in the cache folder, and reuse it afterwards.

    Repeated scans of the hosted files are slow and can hit rate limits, so every heavy query goes
    through here. Delete the cache folder to force a fresh scan.
    """
    path = f"{CACHE}/{name}.parquet"
    if not os.path.exists(path):
        # Write under a temporary name first, so an interrupted run never leaves a half-written cache file.
        con.execute(f"COPY ({sql}) TO '{path}.tmp' (FORMAT parquet)")
        os.replace(f"{path}.tmp", path)
    return con.sql(f"SELECT * FROM read_parquet('{path}')").df()


NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))


show("Connected to the warehouse", {
    "token read from": token_source,
    "token shown anywhere": "no (DuckDB reads it from the environment)",
    "development pairs": ", ".join(f"{a} → {b}" for a, b in PAIRS),
    "sealed pair, never read": "2026-05 → 2026-06",
    "cache folder": CACHE,
})

## Data

The audit uses the same three development pairs, eligibility rules, spread factor, target, and 25 features as the modelling notebook. The sealed May→June pair is added only in section 2b.


**Method:** this cell loads the page-month table for February to May 2026 with the same query and cache as the earlier notebooks, each client's search-history start date, and, from the content table, the content type, the keyword-research fields, and the latest optimization date.


In [ ]:
# The same page-month query and cache as the earlier notebooks.
PAGE_MONTH_SQL = """
    SELECT '{month}' AS month, client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_impressions ELSE 0 END) AS position_impressions,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position ELSE 0 END) AS sum_position,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position * gsc_avg_position ELSE 0 END) AS sum_position_sq,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN 0 ELSE gsc_clicks END) AS clicks_without_position,
           COUNT(*) AS days_with_impressions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS ga4_sessions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions
    FROM {source}
    WHERE gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id"""
page_months = pd.concat([
    cached(f"page_month_v3_{month}", PAGE_MONTH_SQL.format(month=month, source=month_files([month]))) for month in DEV_MONTHS
], ignore_index=True)
assert not page_months.duplicated(["month", "content_hash_id"]).any(), "A page appears twice in one month."

clients = con.sql(f"SELECT client_hash_id, gsc_data_start FROM {DIM_CLIENTS}").df()
clients["gsc_data_start"] = pd.to_datetime(clients["gsc_data_start"])
content = con.sql(f"SELECT content_hash_id, content_type, main_intent, search_volume, competition, cpc, keyword_token_count, last_optimized_date FROM {DIM_CONTENT}").df()
show_table("Inputs loaded", pd.DataFrame({
    "Rows": [f"{len(page_months):,}", f"{len(clients):,}", f"{len(content):,}"],
    "What it is": ["Page-months, February to May 2026", "Clients with their search-history start date",
                   "Pages with content type, keyword research fields, and latest optimization date"],
}, index=pd.Index(["page_months", "clients", "content"], name="Table")))

**Method:** this cell applies the data contract's rules to each development pair (impressions in both months, client history covering all of month M, a valid position in both months, not optimized during the pair). For each month it computes the impression-weighted position, its daily spread, expected clicks from the fine position bins and from the five tiers (both client-scaled), and whether the page fell short by the locked rule: clicks at most half of expected, with negative binomial p below 0.05. The spread factor α is estimated exactly as the data contract does (one estimate per pair from pages whose position barely moved, the median rounded to three decimals), so every notebook shares one target. The month M+1 version of the rule is the target.


In [ ]:
# Rules and bins exactly as in the data contract; the spread factor is estimated below with its method.
BIN_EDGES = [1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 20.5, 50.5]
BIN_NAMES = [str(position) for position in range(1, 11)] + ["11-20", "21-50", "50+"]


def position_bin(position: pd.Series) -> pd.Series:
    """Return each page's position bin: exact rank up to 10, then 11-20, 21-50, and 50+."""
    return pd.Series(np.array(BIN_NAMES)[np.searchsorted(BIN_EDGES, position, side="right")], index=position.index)


def position_tier(position: pd.Series) -> pd.Series:
    """Return the five tiers used by the earlier score: below 3, below 10, below 20, below 50, and 50 or beyond."""
    tiers = np.select([position < 3, position < 10, position < 20, position < 50], ["top_3", "page_1", "striking", "page_3_5"], "deep")
    return pd.Series(tiers, index=position.index)


def client_aware_expected(frame: pd.DataFrame, clicks: str, impressions: str, group: pd.Series) -> pd.Series:
    """Return expected clicks from each group's pooled click rate, scaled by the page's client factor (+1 on both sides)."""
    rate = frame[clicks].groupby(group).sum() / frame[impressions].groupby(group).sum()
    expected = frame[impressions] * group.map(rate)
    by_client = frame.assign(_expected=expected).groupby("client_hash_id")
    factor = (by_client[clicks].sum() + 1) / (by_client["_expected"].sum() + 1)
    return expected * frame["client_hash_id"].map(factor)


def covered_clients(month_m: str) -> pd.Series:
    """Return the clients whose search history starts on or before the first day of month M."""
    return clients.loc[clients["gsc_data_start"] <= pd.Timestamp(f"{month_m}-01"), "client_hash_id"]


def add_month_measures(rows: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """Add position, its spread, click-through, and both expectations for one month."""
    impressions, clicks = rows[f"impressions_{suffix}"], rows[f"clicks_{suffix}"]
    rows[f"position_{suffix}"] = rows[f"sum_position_{suffix}"] / rows[f"position_impressions_{suffix}"]
    spread = rows[f"sum_position_sq_{suffix}"] / rows[f"position_impressions_{suffix}"] - rows[f"position_{suffix}"] ** 2
    rows[f"position_volatility_{suffix}"] = np.sqrt(spread.clip(lower=0))
    rows[f"ctr_{suffix}"] = clicks / impressions
    rows[f"expected_bins_{suffix}"] = client_aware_expected(rows, f"clicks_{suffix}", f"impressions_{suffix}", position_bin(rows[f"position_{suffix}"]))
    rows[f"expected_tier_{suffix}"] = client_aware_expected(rows, f"clicks_{suffix}", f"impressions_{suffix}", position_tier(rows[f"position_{suffix}"]))
    return rows


def add_luck_test(rows: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """Add the negative binomial luck test and the locked shortfall rule for one month."""
    clicks, expected = rows[f"clicks_{suffix}"], rows[f"expected_bins_{suffix}"]
    rows[f"p_{suffix}"] = nbinom.cdf(clicks, 1 / ALPHA, 1 / (1 + ALPHA * expected))
    rows[f"shortfall_{suffix}"] = (rows[f"p_{suffix}"] < 0.05) & (clicks <= 0.5 * expected)
    return rows


def build_pair(month_m: str, month_next: str) -> pd.DataFrame:
    """Return one month pair's eligible rows with both months' measures and the locked target."""
    keys = ["client_hash_id", "content_hash_id"]
    first = page_months[page_months["month"] == month_m].set_index(keys)
    second = page_months[page_months["month"] == month_next].set_index(keys)
    rows = first.join(second, lsuffix="_m", rsuffix="_next", how="inner").reset_index()
    start, end = pd.Timestamp(f"{month_m}-01"), pd.Timestamp(f"{month_next}-01") + pd.offsets.MonthEnd(0)
    rows = rows[rows["client_hash_id"].isin(covered_clients(month_m))]
    rows = rows[(rows["position_impressions_m"] > 0) & (rows["position_impressions_next"] > 0)]
    rows = rows.merge(content, on="content_hash_id", how="left")
    rows = rows[~pd.to_datetime(rows["last_optimized_date"]).between(start, end)].copy()
    rows = add_month_measures(add_month_measures(rows, "m"), "next")
    rows["pair_end"] = end
    return rows.reset_index(drop=True)


def stable_pages(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Return pages whose position moved less than one place, and each one's expected share of clicks in month M+1.

    The expected share corrects for the overall change in click rate between the two months.
    """
    stable = frame[(frame["position_next"] - frame["position_m"]).abs() < 1]
    shift = (stable["clicks_next"].sum() / stable["impressions_next"].sum()) / (stable["clicks_m"].sum() / stable["impressions_m"].sum())
    share = shift * stable["impressions_next"] / (stable["impressions_m"] + shift * stable["impressions_next"])
    return stable, share



def spread_factor(frame: pd.DataFrame) -> float:
    """Return the method-of-moments estimate of the negative binomial spread factor alpha.

    For pages with a stable position, the gap between month-M+1 clicks and the clicks their month-M rate
    predicts has variance (Poisson part) + alpha x (squared clicks). Pooling the excess over the Poisson
    part across pages gives alpha.
    """
    stable, share = stable_pages(frame)
    scale = share / (1 - share)
    gap = stable["clicks_next"] - scale * stable["clicks_m"]
    excess = (gap ** 2 - stable["clicks_next"] - scale ** 2 * stable["clicks_m"]).sum()
    squared = ((stable["clicks_next"] ** 2 - stable["clicks_next"]) + scale ** 2 * (stable["clicks_m"] ** 2 - stable["clicks_m"])).sum()
    return float(excess / squared)



pairs = {f"{month_m} → {month_next}": build_pair(month_m, month_next) for month_m, month_next in PAIRS}
ALPHA = round(float(np.median([spread_factor(rows) for rows in pairs.values()])), 3)
for rows in pairs.values():
    add_luck_test(add_luck_test(rows, "m"), "next")
    rows["target"] = rows["shortfall_next"]
    rows["missed_next"] = (rows["expected_bins_next"] - rows["clicks_next"]).clip(lower=0).where(rows["target"], 0)
show_table(f"Development pairs with the locked target (α = {ALPHA:.3f})", pd.DataFrame({
    "Rows": [f"{len(rows):,}" for rows in pairs.values()],
    "Positives": [f"{rows['target'].sum():,}" for rows in pairs.values()],
    "Base rate": [f"{rows['target'].mean():.1%}" for rows in pairs.values()],
}, index=pd.Index(list(pairs), name="Pair")))

**Method:** this cell builds the modelling notebook's 25 features (the leakage check's 23 plus rule B's two month-M signals) with the same code and checks them.


In [ ]:
# The leakage check's 23 features, plus rule B's two month-M signals.
CONTENT_TYPES = sorted(content["content_type"].dropna().unique())
INTENTS = sorted(content["main_intent"].dropna().unique())
PRODUCT_FIELDS = {"last_optimized_date", "optimization_eligible_date", "is_deleted", "is_published", "content_updated_date"}


def feature_vector(rows: pd.DataFrame) -> pd.DataFrame:
    """Return the model-ready feature matrix for one pair: month-M search features and keyword-research features only."""
    features = pd.DataFrame(index=rows.index)
    features["log_impressions_m"] = np.log1p(rows["impressions_m"])
    features["log_clicks_m"] = np.log1p(rows["clicks_m"])
    features["ctr_m"] = rows["clicks_m"] / rows["impressions_m"]
    features["position_m"] = rows["position_m"]
    features["position_volatility_m"] = rows["position_volatility_m"]
    features["days_with_impressions_m"] = rows["days_with_impressions_m"]
    features["log_expected_m"] = np.log1p(rows["expected_tier_m"])
    for field in ("search_volume", "competition", "cpc", "main_intent"):
        features[f"has_{field}"] = rows[field].notna().astype(int)
    features["log_search_volume"] = np.log1p(rows["search_volume"]).fillna(0)
    features["competition"] = rows["competition"].fillna(0)
    features["log_cpc"] = np.log1p(rows["cpc"]).fillna(0)
    features["keyword_token_count"] = rows["keyword_token_count"]
    for value in CONTENT_TYPES:
        features[f"content_type={value}"] = (rows["content_type"] == value).astype(int)
    for value in INTENTS:
        features[f"intent={value}"] = (rows["main_intent"] == value).astype(int)
    features["intent=unknown"] = rows["main_intent"].isna().astype(int)
    features["log_missed_m"] = np.log1p((rows["expected_bins_m"] - rows["clicks_m"]).clip(lower=0))
    features["short_m"] = rows["shortfall_m"].astype(int)
    return features


vectors = {name: feature_vector(rows) for name, rows in pairs.items()}
FEATURES = list(next(iter(vectors.values())).columns)
for name, features in vectors.items():
    assert list(features.columns) == FEATURES, f"{name}: columns differ from the first pair."
    assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in features.dtypes), f"{name}: a feature is not numeric."
    assert not features.isna().any().any(), f"{name}: a feature still has gaps."
assert not [column for column in FEATURES if column.endswith("_next")], "A month M+1 field reached the features."
assert not set(FEATURES) & PRODUCT_FIELDS, "A product field reached the features."

LABELS = {"log_impressions_m": "Impressions (log)", "log_clicks_m": "Clicks (log)", "ctr_m": "Click-through rate",
          "position_m": "Position", "position_volatility_m": "Position spread", "days_with_impressions_m": "Days with impressions",
          "log_expected_m": "Expected clicks (log)", "has_search_volume": "Has search volume", "has_competition": "Has competition",
          "has_cpc": "Has CPC", "has_main_intent": "Has search intent", "log_search_volume": "Search volume (log)",
          "competition": "Competition", "log_cpc": "CPC (log)", "keyword_token_count": "Keyword length (words)",
          "log_missed_m": "Missed clicks (log)", "short_m": "Short by the target's rule"}


def label(feature: str) -> str:
    """Return a plain-language name for a feature."""
    return LABELS.get(feature) or feature.replace("content_type=", "Content type: ").replace("intent=", "Search intent: ")
show_table("Feature vector, every pair", pd.DataFrame({
    "Rows": [f"{len(features):,}" for features in vectors.values()],
    "Features": [features.shape[1] for features in vectors.values()],
    "Gaps left": [int(features.isna().sum().sum()) for features in vectors.values()],
}, index=pd.Index(list(vectors), name="Pair")))

## 1. Two paper findings + my methodology questions

The FlyRank paper, _The State of AI-Driven SEO_ (March 2026), studies 341,701 pages across 57 brands. Two of its findings bear directly on click-through, the subject of this project. For each, the three questions are the same: where does the number come from, what does the design actually test, and would the claim survive a stricter test? The aim is the next level of rigor, not a scalp; the paper itself flags several of these limits.


### 1a. Finding #3: "Click capture by position tier"

**The claim.** Weighted CTR falls sharply as pages move down the results: 0.423% in the top 3, 0.339% on the rest of page 1, 0.325% at positions 11 to 20, 0.163% at 21 to 50, and 0.050% beyond, an 88% drop. The paper concludes that refining pages already on page 1 is a more reliable source of extra clicks.

**Where the number comes from.** Total clicks ÷ total impressions inside each of five position tiers, over a rolling 90-day window. It is a descriptive aggregate, with no model and no test set.

**My methodology questions.**

1. _Does a tier average hide the shape inside the tier?_ A page's average position blends many searches, and the earlier signal audit found the curve peaking at positions 2 and 3 and dipping near 7, so the top-3 and page-1 averages may each hide a non-monotonic curve.
2. _Is the drop a position effect or a site mix?_ If large sites with low click rates dominate the lower tiers, part of the drop belongs to the sites, not the positions. A per-client spread inside each tier shows how much sites differ.
3. _Does the 90-day position overlap the 90-day clicks?_ Position and clicks come from the same window, so the comparison is fine as a description but cannot say what moving a page up would do.

**Method:** this cell recomputes weighted CTR on every page-month with a valid position, February to May 2026, with the paper's five tiers (using each page's impression-weighted monthly position), then by single position from 1 to 20, and then client by client inside each tier, among clients with at least 10,000 impressions in that tier.


In [ ]:
# The paper's five tiers on the warehouse, then single positions, then the spread across clients.
PAPER_CTR = {"Top 3": 0.00423, "Page 1 (4 to 10)": 0.00339, "Striking (11 to 20)": 0.00325, "Page 3 to 5 (21 to 50)": 0.00163, "Deep (50+)": 0.00050}
visible = page_months[page_months["position_impressions"] > 0].copy()
visible["position"] = visible["sum_position"] / visible["position_impressions"]
visible["tier"] = pd.cut(visible["position"], [0, 3.5, 10.5, 20.5, 50.5, np.inf], labels=list(PAPER_CTR))
tiers = visible.groupby("tier", observed=True)[["clicks", "impressions"]].sum()
tiers["ctr"] = tiers["clicks"] / tiers["impressions"]
show_table("Weighted CTR by the paper's position tiers: paper against warehouse (February to May 2026)", pd.DataFrame({
    "Paper": [f"{PAPER_CTR[tier]:.3%}" for tier in tiers.index],
    "Warehouse": tiers["ctr"].map("{:.3%}".format),
    "Page-months": visible["tier"].value_counts().reindex(tiers.index).map("{:,}".format),
    "Impressions": tiers["impressions"].map("{:,.0f}".format),
}).rename_axis("Tier"))

near = visible[visible["position"] < 20.5].assign(rank=lambda frame: frame["position"].round().clip(lower=1).astype(int))
curve = near.groupby("rank")[["clicks", "impressions"]].sum()
curve["ctr"] = curve["clicks"] / curve["impressions"]
rises = int((curve["ctr"].diff() > 0).sum())
show_table("Weighted CTR by single position, 1 to 20", pd.DataFrame({
    "Weighted CTR": curve["ctr"].map("{:.3%}".format),
    "Change from the position above": curve["ctr"].diff().map(lambda change: "" if pd.isna(change) else f"{change:+.3%}"),
    "Impressions": curve["impressions"].map("{:,.0f}".format),
}).rename_axis("Position"))

per_client = visible.groupby(["tier", "client_hash_id"], observed=True)[["clicks", "impressions"]].sum()
per_client = per_client[per_client["impressions"] >= 10_000]
per_client["ctr"] = per_client["clicks"] / per_client["impressions"]
spread = per_client.groupby(level="tier", observed=True)["ctr"].describe(percentiles=[0.1, 0.5, 0.9])
show_table("Client-level weighted CTR inside each tier (clients with 10,000+ impressions in the tier)", pd.DataFrame({
    "Clients": spread["count"].map("{:.0f}".format),
    "10th percentile": spread["10%"].map("{:.3%}".format),
    "Median": spread["50%"].map("{:.3%}".format),
    "90th percentile": spread["90%"].map("{:.3%}".format),
    "90th ÷ 10th": (spread["90%"] / spread["10%"]).map("{:.1f}x".format),
}).rename_axis("Tier"))
show("Shape of the curve", {"positions 2 to 20 where CTR rises above the position before": f"{rises} of {len(curve) - 1}",
                            "warehouse drop from top 3 to deep": f"{1 - tiers['ctr'].iloc[-1] / tiers['ctr'].iloc[0]:.0%} (paper: 88%)"})

### 1b. ML appendix: "What predicts health?"

**The claim.** A random forest predicts the health score, and position (43%), impressions (32%), scroll depth (15%), and CTR (8%) are its most important features. The model is described as holdout-tested.

**Where the label comes from.** The health score is a FlyRank composite: impressions (30 points) + position (30) + CTR (20) + scroll depth (20). The four top features are the four ingredients of the target, in nearly the order of their weights. The paper says so itself: "health score is partly constructed from inputs such as position and impressions. High importance is therefore expected."

**What the design tests.** An 80/20 random split of 61,790 active pages. Pages of the same brand sit on both sides, and features and target share one window.

**My methodology questions.**

1. _What does a high holdout score mean when the target is a formula of the inputs?_ It shows the model can rebuild the formula, not that it found anything about pages.
2. _Would the score survive a client-grouped split and a future target?_ If the model is asked for next month's health on brands it never saw, the formula can no longer be copied.
3. _How would the importance ranking change?_ With a future target, the ranking would show what predicts health, rather than how health is computed.

**Method:** this cell demonstrates the first two questions on the warehouse. It rebuilds an approximate health score from the paper's formula (impressions, position, and CTR as within-month percentile ranks, weighted 30, 30, and 20; scroll depth is not in the warehouse, so the score runs to 80), then trains the same kind of random forest three ways on the Mar→Apr pair: the paper's design (same-month score, random rows), the same score with clients grouped, and next month's score with clients grouped. It reports R² on the held-out fifth and the forest's built-in importances under the paper's design.


In [ ]:
SEED = 42


def health_like(rows: pd.DataFrame, suffix: str) -> pd.Series:
    """Return an approximate health score from the paper's formula, without scroll depth (maximum 80)."""
    impressions = rows[f"impressions_{suffix}"].rank(pct=True)
    position = 1 - rows[f"position_{suffix}"].rank(pct=True)
    ctr = (rows[f"clicks_{suffix}"] / rows[f"impressions_{suffix}"]).rank(pct=True)
    return 30 * impressions + 30 * position + 20 * ctr


def forest() -> RandomForestRegressor:
    """Return a random forest regressor with fixed settings."""
    return RandomForestRegressor(n_estimators=100, min_samples_leaf=50, max_samples=0.25, n_jobs=-1, random_state=SEED)


names = list(pairs)
demo_name = names[1]
demo_rows, demo_features = pairs[demo_name], vectors[demo_name]
random_fold = next(KFold(n_splits=5, shuffle=True, random_state=SEED).split(demo_features))
client_fold = next(GroupKFold(n_splits=5).split(demo_features, groups=demo_rows["client_hash_id"]))
designs = {"Paper's design: same-month score, random rows": (health_like(demo_rows, "m"), random_fold),
           "Same-month score, clients grouped": (health_like(demo_rows, "m"), client_fold),
           "Next month's score, clients grouped": (health_like(demo_rows, "next"), client_fold)}
fits = {}
for design, (score, (train, test)) in designs.items():
    model = forest().fit(demo_features.iloc[train], score.iloc[train])
    fits[design] = (r2_score(score.iloc[test], model.predict(demo_features.iloc[test])), model)
show_table(f"Predicting an approximate health score, {demo_name} (R² on the held-out fifth)", pd.DataFrame({
    "R²": [f"{r2:.3f}" for r2, _ in fits.values()]}, index=pd.Index(list(fits), name="Design")))
paper_model = fits["Paper's design: same-month score, random rows"][1]
built_in = pd.Series(paper_model.feature_importances_, index=FEATURES).sort_values(ascending=False).head(6)
show_table("Built-in importances under the paper's design (top six)", pd.DataFrame({
    "Importance": built_in.map("{:.0%}".format)}).rename(index=label).rename_axis("Feature"))

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.